In [ ]:
import zarr
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

fig_path = "/home/daniele/atlas_plots_export/"

In [ ]:
atlas = ad.read_zarr("/data/Daniele/atlases/final_versions_topublish/Human.zarr")

## doublet

In [ ]:
macro = atlas[atlas.obs.Level_3.str.contains('Macro')].copy()

In [ ]:
sample_counts = macro.obs["Sample_ID"].value_counts()
valid_samples = sample_counts[sample_counts > 100].index
macro_filtered = macro[macro.obs["Sample_ID"].isin(valid_samples)].copy()

In [ ]:
global_median = macro_filtered.obs["doublet_score"].median()

plt.figure(figsize=(5, 6))

sns.boxplot(
    data=macro_filtered.obs,
    x="Level_4",
    y="doublet_score",
    hue="Level_2",
    palette="Set2",
    showfliers=False,
    legend=False,
)

plt.axhline(y=global_median, color="red", linestyle="-", linewidth=1.5, zorder=2)

plt.text(
    x=4.6, 
    y=global_median, 
    s=f"Median: {global_median:.3f}", 
    color="red", 
    fontsize=10,
    va="bottom"
)

plt.title("doublet scores across Level_4 groups")
plt.xticks(rotation=90, ha="center")
plt.ylim(0, 0.25)
plt.tight_layout()

plt.savefig(f"{fig_path}/doublets/macrophages.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
t_cells = atlas[atlas.obs.Level_3.str.contains('T Ce')].copy()

In [ ]:
sample_counts = t_cells.obs["Sample_ID"].value_counts()
valid_samples = sample_counts[sample_counts > 100].index
t_cells_filtered = t_cells[t_cells.obs["Sample_ID"].isin(valid_samples)].copy()

In [ ]:
global_median = t_cells_filtered.obs["doublet_score"].median()

plt.figure(figsize=(12, 12))

sns.boxplot(
    data=t_cells_filtered.obs,
    x="Level_4",
    y="doublet_score",
    hue="Level_2",
    palette="Set2",
    showfliers=False,
    legend=False,
)
plt.axhline(y=global_median, color="red", linestyle="-", linewidth=1.5, zorder=2)

plt.text(
    x=14.6, 
    y=global_median, 
    s=f"Median: {global_median:.3f}", 
    color="red", 
    fontsize=10,
    va="bottom"
)

plt.title("doublet scores across Level_4 groups")
plt.xticks(rotation=90, ha="center")
plt.ylim(0, 0.25)
plt.tight_layout()

plt.savefig(f"{fig_path}/doublets/t_cells.png", dpi=300, bbox_inches='tight')
plt.show()

## infercnv

In [ ]:
atlas_epi = atlas[
    (atlas.obs["Is_Core"] == "Core")
    & (atlas.obs["Level_2"].isin(["Exocrine Cell", "Malignant Cell"]))
]

In [ ]:
atlas_epi.obs['infercnv'] = atlas_epi.obsm['cnv_score_abs']

In [ ]:
global_median = atlas_epi.obs["infercnv"].median()

plt.figure(figsize=(5, 6))

sns.boxplot(
    data=atlas_epi.obs,
    x="Level_3",
    y="infercnv",
    hue="Level_3",
    palette="Set2",
    showfliers=False,
    legend=False,
)

plt.axhline(y=global_median, color="red", linestyle="-", linewidth=1.5, zorder=2)

plt.text(
    x=4.6, 
    y=global_median, 
    s=f"Median: {global_median:.3f}", 
    color="red", 
    fontsize=10,
    va="bottom"
)

plt.title("")
plt.xticks(rotation=90, ha="center")
plt.ylim(0, 0.05)
plt.tight_layout()

plt.show()

## nb_dist

In [ ]:
import scpdac

In [ ]:
atlas_binned = scpdac.pp.slice_and_bin(atlas, 'human')

In [ ]:
from scipy.optimize import curve_fit
def get_mean_var_df(adata, layer_name):
    X = adata.layers[layer_name]
    if hasattr(X, "toarray"):
        X = X.toarray()
        
    means = np.mean(X, axis=0)
    variances = np.var(X, axis=0)
    
    df = pd.DataFrame({'mean': means, 'variance': variances})
    return df[df['mean'] > 0]
def nb_variance(mu, alpha):
    return mu + alpha * (mu ** 2)

In [ ]:
df_raw = get_mean_var_df(atlas_binned, 'counts')
df_bin = get_mean_var_df(atlas_binned, 'binned_data')

In [ ]:
patient_alphas = []
unique_patients = atlas_binned.obs['Sample_ID'].unique()
for patient in unique_patients:
    adata_sub = atlas_binned[atlas_binned.obs['Sample_ID'] == patient]
    df_patient = get_mean_var_df(adata_sub, 'counts')
    if len(df_patient) > 10:
        try:
            popt, _ = curve_fit(nb_variance, df_patient['mean'], df_patient['variance'], p0=[1.0], maxfev=2000)
            patient_alphas.append(popt[0])
        except:
            pass

In [ ]:
popt_raw, _ = curve_fit(nb_variance, df_raw['mean'], df_raw['variance'], p0=[1.0])
popt_bin, _ = curve_fit(nb_variance, df_bin['mean'], df_bin['variance'], p0=[1.0])
alpha_raw_val = popt_raw[0]
alpha_bin_val = popt_bin[0]
sns.set_theme(style="white", context="talk")
fig, ax = plt.subplots(1, 1, figsize=(11, 7)) 
x_fit_raw = np.logspace(np.log10(df_raw['mean'].min()), np.log10(df_raw['mean'].max()), 500)
y_fit_raw = nb_variance(x_fit_raw, *popt_raw)
ax.plot(x_fit_raw, y_fit_raw, color='#D32F2F', lw=1.5, label='Raw NB Fit')
x_fit_bin = np.logspace(np.log10(df_bin['mean'].min()), np.log10(df_bin['mean'].max()), 500)
y_fit_bin = nb_variance(x_fit_bin, *popt_bin)
ax.plot(x_fit_bin, y_fit_bin, color='#1976D2', lw=1.5, label='Binned NB Fit')
global_min = min(df_raw['mean'].min(), df_bin['mean'].min())
global_max = max(df_raw['mean'].max(), df_bin['mean'].max())
x_identity = np.logspace(np.log10(global_min), np.log10(global_max), 100)
ax.plot(x_identity, x_identity, color='#757575', linestyle='--', lw=1., label='Poisson Reference')

ax.text(x_fit_raw[-1] * 1.15, y_fit_raw[-1], f'$\\alpha$ = {alpha_raw_val:.4f}', 
        color='#D32F2F', va='center', ha='left', fontweight='bold', fontsize=10)

ax.text(x_fit_bin[-1] * 1.15, y_fit_bin[-1], f'$\\alpha$ = {alpha_bin_val:.4f}', 
        color='#1976D2', va='center', ha='left', fontweight='bold', fontsize=10)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Mean Expression ($\mu$)', fontsize=16, labelpad=10)
ax.set_ylabel('Variance ($\sigma^2$)', fontsize=16, labelpad=10)
ax.set_title('')
sns.despine(ax=ax)
ax.tick_params(axis='both', which='major', labelsize=14)

ax.legend(
    bbox_to_anchor=(.7, 0.3), 
    loc='upper left', 
    frameon=False,      
    fontsize=10,
    title='Model Profiles',
    title_fontsize=10
)

plt.tight_layout()
plt.savefig(f"{fig_path}/nb_plots/nb_fit_line.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
sns.set_theme(style="white", context="talk")
fig, ax = plt.subplots(1, 1, figsize=(11, 7))
sns.kdeplot(
    patient_alphas, 
    ax=ax, 
    fill=True, 
    color='#1976D2',
    alpha=0.05, 
    linewidth=2, 
    bw_adjust=0.25
)
sns.rugplot(
    patient_alphas, 
    ax=ax, 
    color='#1976D2', 
    alpha=0.6,
    height=0.03,
    linewidth=1.5
)
ax.axvline(x=alpha_bin_val, color='#1976D2', linestyle='--', linewidth=1.5,           
           label='Mean alpha binned')
ax.axvline(x=alpha_raw_val, color='#D32F2F', linestyle='--', linewidth=1.5,           
           label='Mean alpha raw')
y_max = ax.get_ylim()[1]+.02
text_y_position = y_max * 0.85
ax.text(alpha_bin_val + 0.3, text_y_position, r'$\alpha$ = ' + f'{alpha_bin_val:.4f}', 
        color='#1976D2', va='center', ha='left', fontweight='bold', fontsize=10)

ax.text(alpha_raw_val + 0.3, text_y_position, r'$\alpha$ = ' + f'{alpha_raw_val:.4f}', 
        color='#D32F2F', va='center', ha='left', fontweight='bold', fontsize=10)

ax.set_xlabel(r'Fitted Overdispersion Parameter ($\alpha$)', fontsize=16, labelpad=10)
ax.set_ylabel('Density', fontsize=16, labelpad=10)
ax.set_title('')
sns.despine(ax=ax)
ax.tick_params(axis='both', which='major', labelsize=14)

ax.set_xlim(-10, max(alpha_raw_val, max(patient_alphas)) * 1.15)
ax.legend(
    bbox_to_anchor=(.75, 0.6), 
    loc='upper left', 
    frameon=False,      
    fontsize=14,
    title='Sample Profiles',
    title_fontsize=15
)

plt.tight_layout()
plt.savefig(f"{fig_path}/nb_plots/kde_per_patient.png", dpi=300, bbox_inches='tight')
plt.show()

## spatial composition fig4

In [ ]:
path = "adata.zarr"

def purge_null_encodings(group):
    to_delete = []
    for key, item in group.items():
        if hasattr(item, 'attrs') and item.attrs.get('encoding-type') == 'null':
            print(f"Purging null element: {item.name}")
            to_delete.append(key)
        elif isinstance(item, zarr.hierarchy.Group):
            purge_null_encodings(item)     
    for key in to_delete:
        del group[key]
store = zarr.open(path, mode="r+")
purge_null_encodings(store)
adata = ad.read_zarr(path)

In [ ]:
L4_TO_L3 = {
    "CD4+ Memory T Cell": "CD4+ T Cell",
    "CD4+ Naive T Cell": "CD4+ T Cell",
    "CD4+ Th1 Cell": "CD4+ T Cell",
    "CD4+ Th2 Cell": "CD4+ T Cell",
    "CD4+ Th17 Cell": "CD4+ T Cell",
    "CD4+ Th22 Cell": "CD4+ T Cell",
    "T-reg": "CD4+ T Cell",
    "Double Positive CD4+CD8+ T Cell": "Mixed T Cell",
    "CD8+ Naive T Cell": "CD8+ T Cell",
    "CD8+ Effector T Cell": "CD8+ T Cell",
    "CD8+ Memory T Cell": "CD8+ T Cell",
    "CD8+ Terminal Effector T Cell": "CD8+ T Cell",
    "CD8+ Exhausted T Cell": "CD8+ T Cell",
    "CD8+ Tissue-Resident Memory T Cell": "CD8+ T Cell",
    "γδ T Cell (Vδ1)": "CD4+ T Cell",
    "NK Cell": "NK Cell",
    "B Cell - Naive": "B Cell",
    "B Cell - Memory": "B Cell",
    "B Cell - Germinal Center": "B Cell",
    "B Cell - Activated": "B Cell",
    "B-reg": "B Cell",
    "Plasmablast": "Plasma Cell",
    "Plasma Cell": "Plasma Cell",
    "Dendritic Cell - cDC1": "Dendritic Cell",
    "Dendritic Cell - cDC2": "Dendritic Cell",
    "Dendritic Cell - pDC": "Dendritic Cell",
    "Dendritic Cell - Activated": "Dendritic Cell",
    "Monocyte": "Monocyte",
    "Macrophage - C1Q+ TAM": "Macrophage",
    "Macrophage - CD74+ antigen processing TAM": "Macrophage",
    "Macrophage - lipid processing TAM": "Macrophage",
    "Macrophage - CD3+ TAM": "Macrophage",
    "Macrophage - angiogenic TAM": "Macrophage",
    "Mast Cell": "Mast Cell",
    "Neutrophil - N0": "Neutrophil",
    "Neutrophil - N1": "Neutrophil",
    "Neutrophil - N2": "Neutrophil",
    "Malignant Cell - Epithelial": "Malignant Cell - Epithelial",
    "Malignant Cell - Highly Proliferative": "Malignant Cell - Epithelial",
    "Malignant Cell - EMT": "Malignant Cell - EMT",
    "Malignant Cell - Mesenchymal": "Malignant Cell - Mesenchymal",
    "Malignant Cell - Hypoxia": "Malignant Cell - Epithelial",
    "Malignant Cell - Apoptotic": "Malignant Cell - Epithelial",
    "Malignant Cell - Highly Invasive": "Malignant Cell - Mesenchymal",
    "Malignant Cell - Senescence": "Malignant Cell - Epithelial",
    "Malignant Cell - Acinar-like": "Malignant Cell - Epithelial",
    "Malignant Cell - Pit Like": "Malignant Cell - Epithelial",
    "myCAF": "Cancer Associated Fibroblast",
    "iCAF": "Cancer Associated Fibroblast",
    "Schwann Cell": "Schwann Cell",
    "Smooth Muscle Cell": "Smooth Muscle Cell",
    "Adypocyte": "Adipocyte",
    "Pericyte": "Pericyte",
    "Tumor-Associated Endothelial Cell": "Endothelial Cell",
    "Vascular Endothelial Cell": "Endothelial Cell",
    "Lymphatic Endothelial Cell": "Endothelial Cell",
    "Ductal Cell": "Ductal Cell",
    "Ductal Cell (atypical)": "Ductal Cell (atypical)",
    "Acinar Cell": "Acinar Cell",
    "Acinar (REG+) Cell": "Acinar (REG+) Cell",
    "Acinar Idling Cell": "Acinar Idling Cell",
    "Alpha Cell": "Alpha Cell",
    "Beta Cell": "Beta Cell",
    "Gamma Cell": "Gamma Cell",
    "Delta Cell": "Delta Cell",
    "Epsilon Cell": "Epsilon Cell",
    "Other Endocrine": "Other Endocrine",
}

L3_TO_L2 = {
    "CD4+ T Cell": "Lymphoid",
    "Mixed T Cell": "Lymphoid",
    "CD8+ T Cell": "Lymphoid",
    "NK Cell": "Lymphoid",
    "B Cell": "Lymphoid",
    "Plasma Cell": "Lymphoid",
    "Dendritic Cell": "Myeloid",
    "Monocyte": "Myeloid",
    "Macrophage": "Myeloid",
    "Mast Cell": "Myeloid",
    "Neutrophil": "Myeloid",
    "Malignant Cell - Epithelial": "Malignant Cell",
    "Malignant Cell - EMT": "Malignant Cell",
    "Malignant Cell - Mesenchymal": "Malignant Cell",
    "Cancer Associated Fibroblast": "Stromal Cell",
    "Schwann Cell": "Stromal Cell",
    "Smooth Muscle Cell": "Stromal Cell",
    "Adipocyte": "Stromal Cell",
    "Pericyte": "Stromal Cell",
    "Endothelial Cell": "Endothelial Cell",
    "Ductal Cell": "Exocrine Cell",
    "Ductal Cell (atypical)": "Exocrine Cell",
    "Acinar Cell": "Exocrine Cell",
    "Acinar (REG+) Cell": "Exocrine Cell",
    "Acinar Idling Cell": "Exocrine Cell",
    "Alpha Cell": "Endocrine Cell",
    "Beta Cell": "Endocrine Cell",
    "Gamma Cell": "Endocrine Cell",
    "Delta Cell": "Endocrine Cell",
    "Epsilon Cell": "Endocrine Cell",
    "Other Endocrine": "Endocrine Cell",
}

L2_TO_L1 = {
    "Lymphoid": "Immune Cell",
    "Myeloid": "Immune Cell",
    "Malignant Cell": "Epithelial Cell",
    "Stromal Cell": "Stromal Cell",
    "Endothelial Cell": "Stromal Cell",
    "Exocrine Cell": "Epithelial Cell",
    "Endocrine Cell": "Epithelial Cell",
}

def add_hierarchy_obs(
    adata,
    predicted_col: str = "predicted",
    level4_col: str = "Level_4",
    level3_col: str = "Level_3",
    level2_col: str = "Level_2",
    level1_col: str = "Level_1",
    unknown: str = "Unknown",
):
    """Map Level_4 labels in adata.obs[predicted_col] to Level_3/2/1."""
    if predicted_col not in adata.obs:
        raise KeyError(f"{predicted_col!r} not found in adata.obs")

    l4 = adata.obs[predicted_col].astype("string")

    l3 = l4.map(L4_TO_L3)
    l3 = l3.fillna(unknown)

    l2 = l3.map(L3_TO_L2)
    l2 = l2.fillna(unknown)

    l1 = l2.map(L2_TO_L1)
    l1 = l1.fillna(unknown)
    
    adata.obs[level4_col] = l4.astype("category")
    adata.obs[level3_col] = l3.astype("category")
    adata.obs[level2_col] = l2.astype("category")
    adata.obs[level1_col] = l1.astype("category")

    return adata

In [ ]:
adata = add_hierarchy_obs(adata)

In [ ]:
level_2_palette = {
    "Lymphoid": "#028E95",
    "Myeloid": "#f56d18",
    "Malignant Cell": "#ff0000",
    "Stromal Cell": "#25e907",
    "Endothelial Cell": "#9c0101",
    "Exocrine Cell": "#ea7b95",
    "Endocrine Cell": "#f5d276",
    "Unknown": "black"
}

level_3_palette = {
    "Macrophage": "#8b4513",
    "Acinar (REG+) Cell": "#8a2be2",
    "Malignant Cell - Mesenchymal": "#483d8b",
    "B Cell": "#b03060",
    "Cancer Associated Fibroblast": "#dc143c",
    "Malignant Cell - Epithelial": "#008b8b",
    "Monocyte": "#808000",
    "Epsilon Cell": "#ff8c00",
    "CD4+ T Cell": "#8fbc8f",
    "Ductal Cell": "#ffd700",
    "Ductal Cell (atypical)": "#00ff00",
    "Fibroblast": "#dc143c",
    "NK Cell": "#ff4500",
    "Mixed T Cell": "#800080",
    "Gamma Cell": "#add8e6",
    "CD8+ T Cell": "#00008b",
    "Neutrophil": "#006400",
    "Malignant Cell - EMT": "#bc8f8f",
    "Endothelial Cell": "#dda0dd",
    "Alpha Cell": "#1e90ff",
    "ADM Cell": "#00ff00",
    "Dendritic Cell": "#00fa9a",
    "Beta Cell": "#ff00ff",
    "Acinar Idling Cell": "#dc143c",
    "Delta Cell": "#00bfff",
    "Acinar Cell": "#00ffff",
    "Plasma Cell": "#0000ff",
    "Adipocyte": "#adff2f",
    "Mast Cell": "#ffe4c4",
    "Other Endocrine": "#000000",
    "Pericyte": "#ff1493",
    "Schwann Cell": "#a5a5a5",
    "Smooth Muscle Cell": "#b22222",
}

level_4_palette = {
    'Acinar (REG+) Cell': '#a6cee3',
    'Acinar Cell': '#b2df8a',
    'Acinar Idling Cell': '#fdbf6f',
    'ADM Cell': '#cab2d6',
    'Adipocyte': '#ffff99',
    'Alpha Cell': '#b15928',
    'apCAF': '#1f78b4',
    'B Cell - Activated': '#1f78b4',
    'B Cell - Germinal Center': '#33a02c',
    'B Cell - Memory': '#e31a1c',
    'B Cell - Naive': '#6a3d9a',
    'B-reg': '#b15928',
    'Beta Cell': '#1f78b4',
    'CD4+ Memory T Cell': '#a6cee3',
    'CD4+ Naive T Cell': '#b2df8a',
    'CD4+ Th1 Cell': '#fb9a99',
    'CD4+ Th17 Cell': '#e31a1c',
    'CD4+ Th2 Cell': '#cab2d6',
    'CD4+ Th22 Cell': '#1f78b4',
    'CD8+ Effector T Cell': '#a6cee3',
    'CD8+ Memory T Cell': '#b2df8a',
    'CD8+ Naive T Cell': '#fb9a99',
    'CD8+ Terminal Effector T Cell': '#cab2d6',
    'CD8+ Tissue-Resident Memory T Cell': '#b15928',
    'CD8+ Exhausted T Cell': '#33a02c', 
    'Dendritic Cell - Activated': '#e377c2',
    'Dendritic Cell - cDC1': '#1f78b4',
    'Dendritic Cell - cDC2': '#e31a1c',
    'Dendritic Cell - pDC': '#6a3d9a',
    'Delta Cell': '#bcbd22',
    'Double Positive CD4+CD8+ T Cell': '#e31a1c',
    'Ductal Cell': '#17becf',
    'Ductal Cell (atypical)': '#9edae5',
    'Epsilon Cell': '#393b79',
    'Fibroblast': '#5254a3',
    'Gamma Cell': '#6b6ecf',
    'γδ T Cell (Vδ1)': '#6a3d9a',
    'iCAF': '#9c9ede',
    'Lymphatic Endothelial Cell': '#637939',
    'Macrophage - angiogenic TAM': '#1f78b4',
    'Macrophage - antigen processing TAM': '#ff7f00',
    'Macrophage - CD3+ TAM': '#e31a1c',
    'Macrophage - C1Q+ TAM': '#6a3d9a',
    'Macrophage - lipid processing TAM': '#b15928',
    'Malignant Cell - Pit Like': '#a6cee3',
    'Malignant Cell - Acinar-like': '#b2df8a',
    'Malignant Cell - Epithelial': '#fb9a99',
    'Malignant Cell - Hypoxia': '#fdbf6f',
    'Malignant Cell - EMT': '#cab2d6',
    'Malignant Cell - Highly Proliferative': '#e31a1c',
    'Malignant Cell - Mesenchymal': '#fed683',
    'Malignant Cell - Highly Invasive': '#6a3d9a',
    'Malignant Cell - Senescence': '#1f78b4',
    'Malignant Cell - Apoptotic': '#b15928',
    'Mast Cell': '#393b79',
    'Monocyte': '#6b6ecf',
    'myCAF': '#9c9ede',
    'Neutrophil - N0': '#1f78b4',
    'Neutrophil - N1': '#33a02c',
    'Neutrophil - N2': '#b15928',
    'NK Cell': '#cedb9c',
    'Other Endocrine': '#8c6d31',
    'Pericyte': '#bd9e39',
    'Plasma Cell': '#e7ba52',
    'Plasmablast': '#e7969c',
    'Schwann Cell': '#843c39',
    'Smooth Muscle Cell': '#ad494a',
    'T-reg': '#b15928',
    'Tumor-Associated Endothelial Cell': '#7b4173',
    'Vascular Endothelial Cell': '#a55194'
}

In [ ]:
adata = adata[adata.obs.Level_2 !="Unknown"].copy()

In [ ]:
radio_palette = {'No Radiotherapy': '#a6cee3', 'Radiotherapy': '#e31a1c'}        

In [ ]:
palette = sns.color_palette('Set2')

In [ ]:
def plot_relative_abundances(
    adata,
    custom_cell_types,
    palette,
    cut = True,
    sample_col="sample",
    cell_type_col="Level_2",
    group_col="radiotherapy",
    save=None
):
    radio_palette = palette[:2] if isinstance(palette, list) else palette
    if cut:
        df = adata[adata.obs[cell_type_col].isin(custom_cell_types)].obs[[sample_col, cell_type_col, group_col]].copy()
    else:
        df = adata.obs[[sample_col, cell_type_col, group_col]].copy()

    counts_df = (
        df.groupby([sample_col, group_col, cell_type_col], observed=False)
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    cell_columns = counts_df.columns[2:]
    counts_df = counts_df[counts_df[cell_columns].sum(axis=1) > 0].copy()

    total_cells_per_patient = counts_df[cell_columns].sum(axis=1)
    counts_df[cell_columns] = counts_df[cell_columns].div(
        total_cells_per_patient, axis=0
    )

    available_types = [c for c in custom_cell_types if c in counts_df.columns]
    melted_df = counts_df.melt(
        id_vars=[sample_col, group_col],
        value_vars=available_types,
        var_name="Cell_Type",
        value_name="Relative_Abundance",
    )

    plt.figure(figsize=(10, 6))

    sns.boxplot(
        data=melted_df,
        x="Cell_Type",
        y="Relative_Abundance",
        hue=group_col,
        palette=radio_palette,
        width=0.6,
        fliersize=0,
        boxprops=dict(alpha=0.7, edgecolor="black", linewidth=1),
    )

    sns.stripplot(
        data=melted_df,
        x="Cell_Type",
        y="Relative_Abundance",
        hue=group_col,
        palette={k: "black" for k in melted_df[group_col].unique()},
        dodge=True,
        jitter=0.15,
        size=3,
        edgecolor="black",
        linewidth=0.6,
        alpha=0.9,
        legend=False,
    )

    plt.xlabel("Cell Type", fontsize=12)
    plt.ylabel(
        "Relative Abundance (Proportion of Total Cells)", fontsize=12
    )
    plt.title(
        f"Relative Cell Type Abundances by {group_col.capitalize()} Status",
        fontsize=14,
        pad=15,
    )
    plt.xticks(rotation=15)

    plt.legend(title="Status", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    if save is not None:
        plt.savefig(f"{fig_path}/spatial_compositions/{save}", dpi=300, bbox_inches="tight")

    plt.show()


In [ ]:
plot_relative_abundances(
    adata,
    ['Lymphoid', 'Myeloid', 'Malignant Cell', 'Stromal Cell'],
    palette,
    sample_col="sample",
    cut=False,
    cell_type_col="Level_2",
    group_col="radiotherapy",
    save = "level2.png"
)
plot_relative_abundances(
    adata,
    ['Lymphoid', 'Myeloid', 'Malignant Cell', 'Stromal Cell'],
    palette,
    sample_col="sample",
    cut=True,
    cell_type_col="Level_2",
    group_col="radiotherapy",
    save = "level2_cut.png"
)

In [ ]:
plot_relative_abundances(
    adata,
    ['CD4+ T Cell', 'CD8+ T Cell', 'Macrophage', 'Monocyte', 'Cancer Associated Fibroblast'] ,
    palette,
    sample_col="sample",
    cut=False,
    cell_type_col="Level_3",
    group_col="radiotherapy",
    save = "level3_t_and_caf.png"
)
plot_relative_abundances(
    adata,
    ['CD4+ T Cell', 'CD8+ T Cell', 'Macrophage', 'Monocyte', 'Cancer Associated Fibroblast'] ,
    palette,
    sample_col="sample",
    cut=True,
    cell_type_col="Level_3",
    group_col="radiotherapy",
    save = "level3_t_and_caf_cut.png"
)

In [ ]:
plot_relative_abundances(
    adata,
    [cell for cell in adata.obs.Level_4.unique() if 'Endot' in cell],
    palette,
    sample_col="sample",
    cut=False,
    cell_type_col="Level_4",
    group_col="radiotherapy",
    save = "level3_endo.png"
)
plot_relative_abundances(
    adata,
    [cell for cell in adata.obs.Level_4.unique() if 'Endot' in cell],
    palette,
    sample_col="sample",
    cut=True,
    cell_type_col="Level_4",
    group_col="radiotherapy",
    save = "level3_endo_cut.png"
)

In [ ]:
plot_relative_abundances(
    adata,
    [cell for cell in adata.obs.Level_3.unique() if 'Malignant' in cell],
    palette,
    sample_col="sample",
    cut=False,
    cell_type_col="Level_3",
    group_col="radiotherapy",
    save = "level3_malignant.png"
)
plot_relative_abundances(
    adata,
    [cell for cell in adata.obs.Level_3.unique() if 'Malignant' in cell],
    palette,
    sample_col="sample",
    cut=True,
    cell_type_col="Level_3",
    group_col="radiotherapy",
    save = "level3_malignant_cut.png"
)

In [ ]:
emt_markers = [
    "CDH2",
    "ZEB1",
    "ZEB2",
    "SNAI1",
    "SNAI2",
    "TWIST1",
    "TWIST2",
    "FN1",
    "VIM"
]
emt_markers = [gene for gene in emt_markers if gene in adata.var_names]

In [ ]:
obs = adata[adata.obs.predicted.str.contains("Malignant")].obs[
    ["radiotherapy", "emt_score"]
].copy()

plt.figure(figsize=(3, 6))

sns.boxplot(
    data=obs,
    x="radiotherapy",
    y="emt_score",
    palette="Set2",
    showfliers=False,
)

sns.stripplot(
    data=obs,
    x="radiotherapy",
    y="emt_score",
    color="black",
    size=.3,
    alpha=0.4,
    jitter=0.2,
)

plt.title("EMT Score Distribution by Radiotherapy Status on Malignant Cells", fontsize=10)
plt.xlabel("Radiotherapy", fontsize=10)
plt.ylabel("EMT Score", fontsize=10)
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.savefig(f"/home/daniele/atlas_plots_export/spatial_emt/boxplot_malignant.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
for patient in adata.obs["sample"].unique():
    ad_ = adata[adata.obs["sample"] == patient]
    sc.pl.spatial(
        ad_, 
        spot_size=20, 
        color = "emt_score", 
        show=False, 
        cmap="coolwarm",
        vmax = 6, vmin = -3.5,
        vcenter=0
        
    )    
    plt.savefig(f"/home/daniele/atlas_plots_export/spatial_emt/{patient}spatialt_emt.png", dpi=300, bbox_inches="tight")



## additional celltype

In [ ]:
atlas.obs['EMT score'] = atlas.obsm['EMT score'] 

In [ ]:
sc.pl.dotplot(atlas[atlas.obs.Level_3.str.contains('Malign')], groupby = "Level_3", var_names = "EMT score", standard_scale='var')

In [ ]:
cell_markers = {
    "MAIT_cells": [
        "TRAV1-2",
        "MR1",
        "CCR6",
        "CXCR6",
        "SLC4A10",
        "IL18RAP",
    ],
    "Tuft_cells": [
        "POU2F3",
        "TRPM5",
        "GNAT3",
        "HPGDS",
    ],
    "apCAF": [
        "CD74",
        "HLA-DRA",
        "HLA-DRB1",
        "HLA-DQA1",
    ],
}

In [ ]:
mait_anndata = sc.tl.score_genes(
    atlas[atlas.obs.Level_2 == "Lymphoid"],
    gene_list=cell_markers["MAIT_cells"],
    layer="log_norm",
    score_name="mait_score",
    copy=True,
)

tuft_anndata = sc.tl.score_genes(
    atlas[atlas.obs.Level_2.isin(["Exocrine Cell", "Malignant Cell"])],
    gene_list=cell_markers["Tuft_cells"],
    layer="log_norm",
    score_name="tuft_score",
    copy=True,
)

apcaf_anndata = sc.tl.score_genes(
    atlas[atlas.obs.Level_2 == "Stromal Cell"],
    gene_list=cell_markers["apCAF"],
    layer="log_norm",
    score_name="apcaf_score",
    copy=True,
)

In [ ]:
atlas.obs["Level_5"] = atlas.obs["Level_4"].copy().astype(str)

percentile_threshold = 99.9

score_mapping = {
    "MAIT Cell": mait_anndata.obs["mait_score"],
    "Tuft Cell": tuft_anndata.obs["tuft_score"],
    "apCAF": apcaf_anndata.obs["apcaf_score"],
}

for cell_type, scores in score_mapping.items():
    common_cells = atlas.obs_names.intersection(scores.index)
    if len(common_cells) == 0:
        print(f"WARNING: Zero matching cell barcodes found between atlas and {cell_type}_anndata!")
        continue
        
    valid_scores = scores.loc[common_cells].dropna()
    if len(valid_scores) > 0:
        threshold_value = np.percentile(valid_scores, percentile_threshold)
        passing_cells = valid_scores[valid_scores >= threshold_value].index
        atlas.obs.loc[passing_cells, "Level_5"] = cell_type
    else:
        print(f"WARNING: No valid non-NaN scores for {cell_type}")

atlas.obs["Level_5"] = atlas.obs["Level_5"].astype("category")

In [ ]:
atlas.obs = atlas.obs.copy()

In [ ]:
full_palette = {cat: "#e0e0e0" for cat in atlas.obs["Level_5"].cat.categories}

full_palette["MAIT Cell"] = "red"
full_palette["Tuft Cell"] = "gold"
full_palette["apCAF"] = "blue"

sc.pl.umap(
    atlas, 
    color="Level_5", 
    groups=["MAIT Cell", "Tuft Cell", "apCAF"], 
    palette=full_palette,
    s=5,
    add_outline= True,
    na_color='white',
    frameon=False,
    na_in_legend=False,
    show=False
)
plt.savefig(f"{fig_path}/annotation_addition/umap.png", dpi=300, bbox_inches='tight')

In [ ]:
dp = sc.pl.dotplot(
    atlas[atlas.obs.Level_5.isin(["MAIT Cell","Tuft Cell", "apCAF"])], 
    groupby = "Level_5", var_names = cell_markers, layer='log_norm', return_fig=True
)
dp.add_totals(color="skyblue").style(
    dot_edge_color="black", dot_edge_lw=0.5
).savefig(f"{fig_path}/annotation_addition/dotplot.png", dpi=300, bbox_inches='tight')

In [ ]:
ilc_vs_tcell_markers = {
    "T_Cell": ["CD3D", "CD3E", "CD3G", "TRAC", "TRBC1", "TRBC2"],
    "NK_Cells": ["KLRD1", "NKG7", "GNLY", "PRF1"],
    "Pan_ILC": ["KIT", "IL1RL1",],
    "ILC1": ["TBX21", "EOMES", "IFNG", "CXCR3"],
    "ILC2": ["GATA3", "PTGDR2", "IL17RB", "IL5", "IL13"],
    "ILC3_LTi": ["RORC", "IL23R", "AHR", "IL22"],      
}

In [ ]:
dp = sc.pl.dotplot(
    atlas[atlas.obs.Level_2.isin(["Lymphoid"])], 
    groupby = "Level_3", var_names = ilc_vs_tcell_markers, layer='log_norm', return_fig=True, 
    expression_cutoff=1.25
)
dp.savefig(f"/home/daniele/atlas_new_block/annotation_addition/dotplot.png", dpi=300, bbox_inches='tight')


## adm

In [ ]:
markers_adm = {
    "Acinar": [ "CPB1", "CPA1", "CELA3A", "CELA3B"],
    "Ductal": ["CFTR", "SLC4A4", "EPCAM", "ANXA4"],
}

In [ ]:
dp = sc.pl.dotplot(
    atlas[atlas.obs.Level_2.isin(["Exocrine Cell"])], 
    groupby = "Level_4", var_names = markers_adm, layer='log_norm', return_fig=True, 
    expression_cutoff=1.5, dot_min = 0, dot_max=1
)
dp.savefig(f"/home/daniele/atlas_new_block/annotation_addition/dotplot_exocrine.png", dpi=300, bbox_inches='tight')


## misc

In [ ]:
def plot_relative_abundances(
    adata,
    custom_cell_types,
    palette,
    cut = True,
    sample_col="sample",
    cell_type_col="Level_2",
    group_col="radiotherapy",
    save=None
):
    radio_palette = palette[:2] if isinstance(palette, list) else palette
    if cut:
        df = adata[adata.obs[cell_type_col].isin(custom_cell_types)].obs[[sample_col, cell_type_col, group_col]].copy()
    else:
        df = adata.obs[[sample_col, cell_type_col, group_col]].copy()

    counts_df = (
        df.groupby([sample_col, group_col, cell_type_col], observed=False)
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    cell_columns = counts_df.columns[2:]
    counts_df = counts_df[counts_df[cell_columns].sum(axis=1) > 0].copy()

    total_cells_per_patient = counts_df[cell_columns].sum(axis=1)
    counts_df[cell_columns] = counts_df[cell_columns].div(
        total_cells_per_patient, axis=0
    )

    available_types = [c for c in custom_cell_types if c in counts_df.columns]
    melted_df = counts_df.melt(
        id_vars=[sample_col, group_col],
        value_vars=available_types,
        var_name="Cell_Type",
        value_name="Relative_Abundance",
    )

    plt.figure(figsize=(10, 6))

    sns.boxplot(
        data=melted_df,
        x="Cell_Type",
        y="Relative_Abundance",
        hue=group_col,
        palette=radio_palette,
        width=0.6,
        fliersize=0,
        boxprops=dict(alpha=0.7, edgecolor="black", linewidth=1),
    )

    sns.stripplot(
        data=melted_df,
        x="Cell_Type",
        y="Relative_Abundance",
        hue=group_col,
        palette={k: "black" for k in melted_df[group_col].unique()},
        dodge=True,
        jitter=0.15,
        size=.03,
        edgecolor="black",
        linewidth=0.6,
        alpha=0.9,
        legend=False,
    )

    plt.xlabel("Cell Type", fontsize=12)
    plt.ylabel(
        "Relative Abundance (Proportion of Total Cells)", fontsize=12
    )
    plt.title(
        f"Relative Cell Type Abundances by {group_col.capitalize()} Status",
        fontsize=14,
        pad=15,
    )
    plt.xticks(rotation=90)

    plt.legend(title="Status", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    if save is not None:
        plt.savefig(f"{fig_path}/additional_misc/{save}", dpi=300, bbox_inches="tight")

    plt.show()


In [ ]:
palette = { 
    'Male': "#56B4E9",
    'Female': "#CC79A7",
    'Unknown': "#BBBBBB"
}
plot_relative_abundances(
    atlas[atlas.obs.Sex!='Unknown'],
    atlas.obs.Level_3.unique().tolist() ,
    palette,
    sample_col="Sample_ID",
    cut=False,
    cell_type_col="Level_3",
    group_col="Sex",
    save = "sex_level_3.png"
)

In [ ]:
differentiation_markers = {
    "Normal Ductal": ["CFTR", "BICC1", 'PKHD1'],
    "Atypical / Stress Bridge": ["ANXA4", "CLU", "FXYD2"],
    "Fully Malignant": ["CRIP1", "IFI27", "S100A6", "S100P"]
}
sc.pl.dotplot(
    atlas[atlas.obs.Level_3.isin(['Ductal Cell', 'Ductal Cell (atypical)', 'Malignant Cell - Epithelial' ])], 
    expression_cutoff = 0.1, 
    var_names = differentiation_markers, 
    groupby = "Level_3",
    layer='log_norm',
    show=False
)
plt.savefig(f"{fig_path}/additional_misc/dotplot_malignant_markers.png", dpi=300, bbox_inches='tight')

In [ ]:
sc.pl.umap(
    atlas, 
    color="Tissue", 
    add_outline= True,
    na_color='white',
    frameon=False,
    na_in_legend=False,
    show=False
)
plt.savefig(f"{fig_path}/additional_misc/Tissue.png", dpi=300, bbox_inches='tight')

In [ ]:
sc.pl.umap(
    atlas, 
    color="Condition", 
    add_outline= True,
    na_color='white',
    frameon=False,
    na_in_legend=False,
    show=False
)
plt.savefig(f"{fig_path}/additional_misc/Condition.png", dpi=300, bbox_inches='tight')

In [ ]:
def get_clean_palette(adata, column_name):
    categories = adata.obs[column_name].astype('category').cat.categories.tolist()
    color_key = f"{column_name}_colors"
    
    if color_key in adata.uns:
        colors = list(adata.uns[color_key])
        if len(colors) == len(categories):
            return {str(cat): col for cat, col in zip(categories, colors)}
            
    fallback_colors = sns.color_palette("tab20", len(categories))
    return {str(cat): col for cat, col in zip(categories, fallback_colors)}

tissue_palette = get_clean_palette(atlas, "Tissue")
condition_palette = get_clean_palette(atlas, "Condition")

plt.figure(figsize=(6, 4))
tissue_counts = atlas.obs["Tissue"].value_counts().reset_index()
tissue_counts.columns = ["Tissue", "Count"]
tissue_counts["Tissue"] = tissue_counts["Tissue"].astype(str)

sns.barplot(
    data=tissue_counts, 
    x="Tissue", 
    y="Count", 
    palette=tissue_palette, 
    hue="Tissue", 
    legend=False
)
plt.title("Number of Cells per Tissue", fontweight="bold", pad=15)
plt.ylabel("Cell Count")
plt.xlabel("Tissue")
plt.xticks(rotation=45, ha="right")
plt.savefig(f"{fig_path}/additional_misc/Tissue_cells_numbers.png", dpi=300, bbox_inches='tight')
plt.close()

plt.figure(figsize=(6, 4))
tissue_samples = atlas.obs.groupby("Tissue", observed=False)["Sample_ID"].nunique().reset_index()
tissue_samples.columns = ["Tissue", "Unique_Samples"]
tissue_samples["Tissue"] = tissue_samples["Tissue"].astype(str)

sns.barplot(
    data=tissue_samples, 
    x="Tissue", 
    y="Unique_Samples", 
    palette=tissue_palette, 
    hue="Tissue", 
    legend=False
)
plt.title("Number of Unique Samples per Tissue", fontweight="bold", pad=15)
plt.ylabel("Sample Count")
plt.xlabel("Tissue")
plt.xticks(rotation=45, ha="right")
plt.gca().yaxis.get_major_locator().set_params(integer=True)
plt.savefig(f"{fig_path}/additional_misc/Tissue_sample_numbers.png", dpi=300, bbox_inches='tight')
plt.close()

plt.figure(figsize=(6, 4))
condition_counts = atlas.obs["Condition"].value_counts().reset_index()
condition_counts.columns = ["Condition", "Count"]
condition_counts["Condition"] = condition_counts["Condition"].astype(str)

sns.barplot(
    data=condition_counts, 
    x="Condition", 
    y="Count", 
    palette=condition_palette, 
    hue="Condition", 
    legend=False
)
plt.title("Number of Cells per Condition", fontweight="bold", pad=15)
plt.ylabel("Cell Count")
plt.xlabel("Condition")
plt.xticks(rotation=45, ha="right")
plt.savefig(f"{fig_path}/additional_misc/Condition_cells_numbers.png", dpi=300, bbox_inches='tight')
plt.close()

plt.figure(figsize=(6, 4))
condition_samples = atlas.obs.groupby("Condition", observed=False)["Sample_ID"].nunique().reset_index()
condition_samples.columns = ["Condition", "Unique_Samples"]
condition_samples["Condition"] = condition_samples["Condition"].astype(str)

sns.barplot(
    data=condition_samples, 
    x="Condition", 
    y="Unique_Samples", 
    palette=condition_palette, 
    hue="Condition", 
    legend=False
)
plt.title("Number of Unique Samples per Condition", fontweight="bold", pad=15)
plt.ylabel("Sample Count")
plt.xlabel("Condition")
plt.xticks(rotation=45, ha="right")
plt.gca().yaxis.get_major_locator().set_params(integer=True)
plt.savefig(f"{fig_path}/additional_misc/Condition_sample_numbers.png", dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
level_4_series = atlas.obs['Level_4']
sample_id_series = atlas.obs['Sample_ID']

malignant_mask = level_4_series.str.contains('Malignant', na=False)
filtered_level_4 = level_4_series[malignant_mask]
filtered_sample_id = sample_id_series[malignant_mask]

cross_tab = pd.crosstab(filtered_level_4, filtered_sample_id)

del filtered_level_4
del filtered_sample_id
del malignant_mask

relative_abundance = cross_tab.div(cross_tab.sum(axis=0), axis=1) * 100
patient_counts_above_threshold = (relative_abundance > 1.0).sum(axis=1)

annotation_cmap = plt.cm.binary
v_min, v_max = patient_counts_above_threshold.min(), patient_counts_above_threshold.max()
norm_counts = (patient_counts_above_threshold - v_min) / (v_max - v_min) if v_max > v_min else patient_counts_above_threshold
row_colors_series = pd.Series([annotation_cmap(0.1 + 0.6 * val) for val in norm_counts], index=patient_counts_above_threshold.index)

g = sns.clustermap(
    relative_abundance,
    cmap='vlag',
    row_colors=row_colors_series,
    row_cluster=True,
    metric='euclidean',
    method='ward',
    xticklabels=False,
    figsize=(20, 10),
    colors_ratio=0.015,
    cbar_pos=(0.35, -0.02, 0.3, 0.02),
    cbar_kws={'label': "Relative Abundance (% of Patient's Malignant Cells)", 'orientation': 'horizontal'}
)

g.ax_row_dendrogram.set_visible(False)

plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0, fontsize=10)

reordered_rows = g.dendrogram_row.reordered_ind
for i, idx in enumerate(reordered_rows):
    program_name = relative_abundance.index[idx]
    count_val = patient_counts_above_threshold.loc[program_name]
    g.ax_row_colors.text(-0.5, i + 0.5, f"n={count_val} (>1%)", 
                         ha='right', va='center', fontsize=9, fontweight='bold', color='black')

plt.savefig(f"{fig_path}/additional_misc/clustermap_malignant_states.png", dpi=300, bbox_inches='tight')
plt.show()

## crosstab

In [ ]:
def get_cell_crosstab(
    adata, group_cols=["Level_4", "Condition", "Treatment", "Tissue", "Sex"]
):
    """Generates a hierarchical/flat cell-count table for N metadata columns."""
    cell_counts = (
        adata.obs.groupby(group_cols, observed=False)
        .size()
        .reset_index(name="Cell_Count")
    )
    cell_counts = cell_counts[cell_counts["Cell_Count"] > 0].reset_index(
        drop=True
    )
    total_cells = len(adata)
    cell_counts["Cell_Percentage (%)"] = (
        (cell_counts["Cell_Count"] / total_cells) * 100
    ).round(2)

    return cell_counts.sort_values(by="Cell_Count", ascending=False)

In [ ]:
def get_sample_crosstab(
    adata,
    group_cols=["Level_4", "Condition", "Treatment", "Tissue", "Sex"],
    sample_col="Sample_ID",
):
    """Generates a sample-level count table for N metadata columns."""
    for col in adata.obs.columns:
        if adata.obs[col].dtype == "category":
            if "Unknown" not in adata.obs[col].cat.categories:
                adata.obs[col] = adata.obs[col].cat.add_categories("Unknown")
            adata.obs[col] = adata.obs[col].fillna("Unknown")
        elif adata.obs[col].dtype == "object":
            adata.obs[col] = adata.obs[col].fillna("Unknown")
    sample_counts = (
        adata.obs.groupby(group_cols, observed=False)[sample_col]
        .nunique()
        .reset_index(name="Sample_Count")
    )
    sample_counts = sample_counts[sample_counts["Sample_Count"] > 0].reset_index(
        drop=True
    )
    total_samples = len(adata.obs[sample_col].unique())

    sample_counts["Atlas_Percentage (%)"] = (
        (sample_counts["Sample_Count"] / total_samples) * 100
    ).round(2)
    return sample_counts.sort_values(by="Sample_Count", ascending=False)

In [ ]:
metadata = [
    "Dataset",
    "Technology",
    "Age",
    "Sex",
    "Treatment",
    "TreatmentType",
    "TreatmentStatus",
    "Tissue",
    "Condition",
    "Diabetes",
]

In [ ]:
cell_crosstab = get_cell_crosstab(atlas, group_cols = metadata)
sample_crosstab = get_sample_crosstab(atlas, group_cols = metadata)

In [ ]:
cell_crosstab.to_csv("/home/daniele/metadata_crosstab/metadata_cell_crosstab_human.csv", index=False)
sample_crosstab.to_csv("/home/daniele/metadata_crosstab/metadata_sample_crosstab_human.csv", index=False)

In [ ]:
atlas = ad.read_zarr("/data/Daniele/atlases/final_versions_topublish/Mouse.zarr")

In [ ]:
metadata_mouse = [
    "Level_4",
    "Dataset",
    "Technology",
    "Sex",
    "Treatment",
    "TreatmentType",
    "TreatmentStatus",
    "Tissue",
    "Model",
    "Barcoded",
]

In [ ]:
sample_crosstab = get_sample_crosstab(atlas, group_cols = metadata_mouse[1:])

In [ ]:
sample_crosstab.to_csv("/home/daniele/metadata_crosstab/metadata_sample_crosstab_mouse.csv", index=False)